# Supplementary marker discovery

Identify genes enriched in the atlas-defined **trichome cell** annotation and compare them with the curated trichome modules used in the main notebook.


## Load atlas

The notebook expects the project structure to contain `data/` and `tables/` folders.


In [2]:
from pathlib import Path
import os

import scanpy as sc
import pandas as pd
import numpy as np


def find_project_dir():
    """Find repository root from root, notebooks/, or scripts/."""
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "data" / "final_module_definitions.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find repository root. Open the project folder or set PROJECT_DIR manually."
    )


PROJECT_DIR = find_project_dir()
os.chdir(PROJECT_DIR)

DATA_DIR = Path("data")
TABLE_DIR = Path("tables")
TABLE_DIR.mkdir(exist_ok=True)

adata = sc.read_h5ad(DATA_DIR / "E-ENAD-53.project.h5ad")

print("Project directory:", PROJECT_DIR)
adata


Project directory: c:\Users\fforn\tomato_shoot_apex_atlas


AnnData object with n_obs × n_vars = 20278 × 26293
    obs: 'age', 'cultivar', 'developmental_stage', 'genotype', 'growth_condition', 'organism_part', 'organism', 'authors_cell_type_-_ontology_labels', 'authors_cell_type', 'organism_part.1', 'age_ontology', 'cultivar_ontology', 'developmental_stage_ontology', 'genotype_ontology', 'growth_condition_ontology', 'organism_part_ontology', 'organism_ontology', 'authors_cell_type_-_ontology_labels_ontology', 'authors_cell_type_ontology', 'organism_part_ontology.1', 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'n_counts', 'n_genes', 'louvain_resolution_0.1', 'louvain_resolution_0.3', 'louvain_resolution_0.5', 'louvain_resolution_0.7', 'louvain_resolution_1.0', 'louvain_resolution_2.0', 'louvain_resolution_3.0', 'louvain_resolution_4.0', 'louvain_resolution_5.0'
    var: 'gene_symbols', 'chromosome', 

## Select trichome annotation

The authors' fine cell-type annotation contains the `trichome cell` label.


In [3]:
label_col = "authors_cell_type"
target_label = "trichome cell"

cell_counts = adata.obs[label_col].value_counts()
print(cell_counts)

n_target = adata.obs[label_col].eq(target_label).sum()
n_total = adata.n_obs

print()
print(f"Target label: {target_label}")
print(f"Trichome nuclei: {n_target}")
print(f"Total nuclei: {n_total}")
print(f"Percentage: {n_target / n_total * 100:.2f}%")


authors_cell_type
cell cycle S phase-1       1794
mesophyll-adaxial          1753
epidermis-late             1611
meristem                   1207
vascular leaf-1            1085
cell cycle G2/M phase-1    1057
cell cycle G2/M phase-2     957
epidermis-early             692
cell cycle S phase-2        647
subepidermis-abaxial        645
trichome cell               515
developing xylem            477
mesophyll-abaxial           464
vascular leaf-2             288
developing phloem            74
Name: count, dtype: int64

Target label: trichome cell
Trichome nuclei: 515
Total nuclei: 20278
Percentage: 2.54%


## Rank trichome-enriched genes

Trichome-labelled nuclei are compared against all other atlas nuclei using a Wilcoxon rank-sum test.


In [4]:
sc.tl.rank_genes_groups(
    adata,
    groupby=label_col,
    groups=[target_label],
    reference="rest",
    method="wilcoxon",
    key_added="rank_genes_trichome_vs_rest",
)

trichome_markers = sc.get.rank_genes_groups_df(
    adata,
    group=target_label,
    key="rank_genes_trichome_vs_rest",
)

trichome_markers.head(20)


,names,scores,logfoldchanges,pvals,pvals_adj
0,Solyc10g075100.2,35.847713,22.434286,1.996272e-281,7.151643e-277
1,Solyc09g092710.3,31.749567,10.393949,3.219245e-221,5.766473e-217
2,Solyc11g006250.2,30.969770,9.710872,1.376701e-210,1.644010e-206
3,Solyc07g055950.3,28.198473,13.951530,6.105243e-175,5.468009e-171
4,Solyc09g097780.3,27.060371,9.505426,2.883819e-161,2.066256e-157
5,Solyc03g121180.3,26.581535,6.976787,1.109868e-155,6.626836e-152
6,Solyc09g062970.1,26.480524,28.456154,1.624815e-154,8.315572e-151
7,Solyc03g121590.3,26.375414,8.220777,2.623788e-153,1.174965e-149
8,Solyc01g005990.3,26.205847,6.190680,2.279414e-151,9.073335e-148
9,Solyc07g006300.3,25.720915,6.337602,6.821559e-146,2.443823e-142


## Add detection summaries

Detection summaries are calculated for the top 1,000 ranked genes to support high-confidence filtering.


In [5]:
X = adata.X
gene_names = pd.Index(adata.var_names)

target_mask = adata.obs[label_col].eq(target_label).values
rest_mask = ~target_mask


def get_expression_vector(gene):
    """Return expression values for one atlas gene."""
    idx = gene_names.get_loc(gene)
    expr = X[:, idx]
    if hasattr(expr, "toarray"):
        return expr.toarray().ravel()
    return np.asarray(expr).ravel()


summary_rows = []

for gene in trichome_markers["names"].head(1000):
    expr = get_expression_vector(gene)
    expr_target = expr[target_mask]
    expr_rest = expr[rest_mask]

    summary_rows.append({
        "gene": gene,
        "mean_trichome": expr_target.mean(),
        "mean_rest": expr_rest.mean(),
        "pct_detected_trichome": (expr_target > 0).mean() * 100,
        "pct_detected_rest": (expr_rest > 0).mean() * 100,
    })

expr_summary = pd.DataFrame(summary_rows)
expr_summary.head()


,gene,mean_trichome,mean_rest,pct_detected_trichome,pct_detected_rest
0,Solyc10g075100.2,1.749747,-0.045596,98.640777,31.629813
1,Solyc09g092710.3,2.138510,-0.055727,88.349515,15.053383
2,Solyc11g006250.2,1.838610,-0.047912,87.961165,20.062744
3,Solyc07g055950.3,1.220229,-0.031798,86.796117,37.919344
4,Solyc09g097780.3,1.372190,-0.035758,84.660194,27.683044


## Export trichome marker rankings

The full ranking is exported; detection columns are filled for the top 1,000 ranked genes.


In [6]:
trichome_markers_export = (
    trichome_markers
    .rename(columns={"names": "gene"})
    .merge(expr_summary, on="gene", how="left")
    .sort_values(["pvals_adj", "scores"], ascending=[True, False])
)

trichome_markers_export["solyc_id_base"] = trichome_markers_export["gene"].str.replace(
    r"\.\d+$", "", regex=True
)

full_out = TABLE_DIR / "atlas_trichome_cell_markers_vs_rest.csv"
trichome_markers_export.to_csv(full_out, index=False)

print(f"Tested genes: {len(trichome_markers_export):,}")
print(f"Saved: {full_out}")
trichome_markers_export.head(30)


Tested genes: 35,825
Saved: tables\atlas_trichome_cell_markers_vs_rest.csv


,gene,scores,logfoldchanges,pvals,pvals_adj,mean_trichome,mean_rest,pct_detected_trichome,pct_detected_rest,solyc_id_base
0,Solyc10g075100.2,35.847713,22.434286,1.996272e-281,7.151643e-277,1.749747,-0.045596,98.640777,31.629813,Solyc10g075100
1,Solyc09g092710.3,31.749567,10.393949,3.219245e-221,5.766473e-217,2.138510,-0.055727,88.349515,15.053383,Solyc09g092710
2,Solyc11g006250.2,30.969770,9.710872,1.376701e-210,1.644010e-206,1.838610,-0.047912,87.961165,20.062744,Solyc11g006250
3,Solyc07g055950.3,28.198473,13.951530,6.105243e-175,5.468009e-171,1.220229,-0.031798,86.796117,37.919344,Solyc07g055950
4,Solyc09g097780.3,27.060371,9.505426,2.883819e-161,2.066256e-157,1.372190,-0.035758,84.660194,27.683044,Solyc09g097780
5,Solyc03g121180.3,26.581535,6.976787,1.109868e-155,6.626836e-152,1.926216,-0.050195,75.728155,12.396903,Solyc03g121180
6,Solyc09g062970.1,26.480524,28.456154,1.624815e-154,8.315572e-151,1.031545,-0.026881,89.320388,48.089865,Solyc09g062970
7,Solyc03g121590.3,26.375414,8.220777,2.623788e-153,1.174965e-149,1.192637,-0.031079,86.213592,34.387492,Solyc03g121590
8,Solyc01g005990.3,26.205847,6.190680,2.279414e-151,9.073335e-148,2.036044,-0.053057,74.174757,10.317260,Solyc01g005990
9,Solyc07g006300.3,25.720915,6.337602,6.821559e-146,2.443823e-142,1.855505,-0.048352,74.368932,12.391843,Solyc07g006300


## Define high-confidence trichome markers

A stricter subset keeps genes with positive enrichment, FDR < 0.05, and >10% detection in trichome-labelled nuclei.


In [7]:
sig_trichome_markers = trichome_markers_export[
    (trichome_markers_export["scores"] > 0) &
    (trichome_markers_export["pvals_adj"] < 0.05)
].copy()

high_confidence = trichome_markers_export[
    (trichome_markers_export["scores"] > 5) &
    (trichome_markers_export["pct_detected_trichome"] > 10) &
    (trichome_markers_export["pvals_adj"] < 0.05)
].copy()

hc_out = TABLE_DIR / "author_high_confidence_trichome_markers.csv"
high_confidence.to_csv(hc_out, index=False)

print(f"Significantly enriched genes: {len(sig_trichome_markers):,}")
print(f"High-confidence trichome markers: {len(high_confidence):,}")
print(f"Saved: {hc_out}")

high_confidence.head(30)


Significantly enriched genes: 1,695
High-confidence trichome markers: 651
Saved: tables\author_high_confidence_trichome_markers.csv


,gene,scores,logfoldchanges,pvals,pvals_adj,mean_trichome,mean_rest,pct_detected_trichome,pct_detected_rest,solyc_id_base
0,Solyc10g075100.2,35.847713,22.434286,1.996272e-281,7.151643e-277,1.749747,-0.045596,98.640777,31.629813,Solyc10g075100
1,Solyc09g092710.3,31.749567,10.393949,3.219245e-221,5.766473e-217,2.138510,-0.055727,88.349515,15.053383,Solyc09g092710
2,Solyc11g006250.2,30.969770,9.710872,1.376701e-210,1.644010e-206,1.838610,-0.047912,87.961165,20.062744,Solyc11g006250
3,Solyc07g055950.3,28.198473,13.951530,6.105243e-175,5.468009e-171,1.220229,-0.031798,86.796117,37.919344,Solyc07g055950
4,Solyc09g097780.3,27.060371,9.505426,2.883819e-161,2.066256e-157,1.372190,-0.035758,84.660194,27.683044,Solyc09g097780
5,Solyc03g121180.3,26.581535,6.976787,1.109868e-155,6.626836e-152,1.926216,-0.050195,75.728155,12.396903,Solyc03g121180
6,Solyc09g062970.1,26.480524,28.456154,1.624815e-154,8.315572e-151,1.031545,-0.026881,89.320388,48.089865,Solyc09g062970
7,Solyc03g121590.3,26.375414,8.220777,2.623788e-153,1.174965e-149,1.192637,-0.031079,86.213592,34.387492,Solyc03g121590
8,Solyc01g005990.3,26.205847,6.190680,2.279414e-151,9.073335e-148,2.036044,-0.053057,74.174757,10.317260,Solyc01g005990
9,Solyc07g006300.3,25.720915,6.337602,6.821559e-146,2.443823e-142,1.855505,-0.048352,74.368932,12.391843,Solyc07g006300


## Compare final modules with atlas-derived markers

Curated module genes are matched to the atlas ranking after removing Solyc version suffixes.


In [8]:
module_file = DATA_DIR / "final_module_definitions.csv"
modules = pd.read_csv(module_file)

modules["solyc_id_base"] = modules["solyc_id"].str.replace(r"\.\d+$", "", regex=True)
modules.head()


,gene_name,display_name,solyc_id,module,module_label,module_order,gene_order,gene_role,functional_role,function_summary,reference_label,notes,solyc_id_base
0,ML1,ML1,Solyc10g005330,epidermal_identity,Epidermal identity,1,1,HD-Zip IV transcription factor,Epidermal/L1 identity marker,Meristem Layer 1 marker used for epidermal/pro...,Tian et al. 2020; epidermal marker literature,NaN,Solyc10g005330
1,PDF1,PDF1,Solyc07g055950,epidermal_identity,Epidermal identity,1,2,Protodermal factor,Epidermal/protoderm marker,Protodermal factor linked to epidermal identit...,Tian et al. 2020,NaN,Solyc07g055950
2,CD2,CD2,Solyc01g091630,epidermal_identity,Epidermal identity,1,3,HD-Zip IV / cuticle regulator,Epidermal/cuticle context,Cuticle and epidermal regulator used as epider...,Tian et al. 2020; Galdon-Armero 2018; Zocca et...,Same locus also reported as CD2/ANL2 in atlas ...,Solyc01g091630
3,SVB,SVB,Solyc01g100750,epidermal_identity,Epidermal identity,1,4,Epidermis/trichome developmental regulator,Epidermal-trichome GRN node,SMALLER TRICHOMES; atlas epidermis-trichome re...,Tian et al. 2020,NaN,Solyc01g100750
4,WOOLLY / Wo,Wo,Solyc02g080260,epidermal_identity,Epidermal identity,1,5,HD-Zip IV transcription factor,Epidermal-to-trichome competence regulator,Master regulator of trichome initiation and mo...,Tian et al. 2020; Wu et al. 2023; Zocca et al....,Bridge marker; not interpreted as purely epide...,Solyc02g080260


## Add module-gene detection summaries

Detection is calculated for every final module gene, including weakly expressed metabolic markers.


In [9]:
module_detection_rows = []

for _, row in modules.iterrows():
    solyc_base = row["solyc_id_base"]
    matches = gene_names[gene_names.str.replace(r"\.\d+$", "", regex=True) == solyc_base]

    if len(matches) == 0:
        module_detection_rows.append({
            "solyc_id_base": solyc_base,
            "gene_atlas": np.nan,
            "mean_trichome": np.nan,
            "mean_rest": np.nan,
            "pct_detected_trichome": np.nan,
            "pct_detected_rest": np.nan,
        })
        continue

    gene_atlas = matches[0]
    expr = get_expression_vector(gene_atlas)
    expr_target = expr[target_mask]
    expr_rest = expr[rest_mask]

    module_detection_rows.append({
        "solyc_id_base": solyc_base,
        "gene_atlas": gene_atlas,
        "mean_trichome": expr_target.mean(),
        "mean_rest": expr_rest.mean(),
        "pct_detected_trichome": (expr_target > 0).mean() * 100,
        "pct_detected_rest": (expr_rest > 0).mean() * 100,
    })

module_detection = pd.DataFrame(module_detection_rows)
module_detection.head()


,solyc_id_base,gene_atlas,mean_trichome,mean_rest,pct_detected_trichome,pct_detected_rest
0,Solyc10g005330,Solyc10g005330.3,0.542701,-0.014142,36.504854,15.341800
1,Solyc07g055950,Solyc07g055950.3,1.220229,-0.031798,86.796117,37.919344
2,Solyc01g091630,Solyc01g091630.3,0.813039,-0.021187,73.980583,33.987755
3,Solyc01g100750,Solyc01g100750.2,1.168669,-0.030454,39.805825,7.741740
4,Solyc02g080260,Solyc02g080260.3,0.590130,-0.015378,22.912621,6.790467


## Export module support table

`atlas_trichome_supported` marks module genes significantly enriched in trichome-labelled nuclei.


In [10]:
module_marker_overlap = modules.merge(
    trichome_markers_export[
        ["solyc_id_base", "scores", "logfoldchanges", "pvals", "pvals_adj"]
    ],
    on="solyc_id_base",
    how="left",
).merge(
    module_detection,
    on="solyc_id_base",
    how="left",
)

module_marker_overlap["atlas_trichome_supported"] = (
    (module_marker_overlap["scores"] > 0) &
    (module_marker_overlap["pvals_adj"] < 0.05)
)

sort_cols = [c for c in ["module_order", "gene_order"] if c in module_marker_overlap.columns]
if sort_cols:
    module_marker_overlap = module_marker_overlap.sort_values(sort_cols)

keep_cols = [
    "module_label",
    "display_name",
    "solyc_id",
    "gene_atlas",
    "gene_role",
    "atlas_trichome_supported",
    "scores",
    "logfoldchanges",
    "pvals_adj",
    "mean_trichome",
    "mean_rest",
    "pct_detected_trichome",
    "pct_detected_rest",
]

module_marker_overlap_simple = module_marker_overlap[keep_cols]

module_out = TABLE_DIR / "final_modules_vs_atlas_trichome_markers.csv"
module_marker_overlap_simple.to_csv(module_out, index=False)

print(f"Saved: {module_out}")
module_marker_overlap_simple


Saved: tables\final_modules_vs_atlas_trichome_markers.csv


,module_label,display_name,solyc_id,gene_atlas,gene_role,atlas_trichome_supported,scores,logfoldchanges,pvals_adj,mean_trichome,mean_rest,pct_detected_trichome,pct_detected_rest
0,Epidermal identity,ML1,Solyc10g005330,Solyc10g005330.3,HD-Zip IV transcription factor,True,8.502211,2.183427,2.988326e-15,0.542701,-0.014142,36.504854,15.341800
1,Epidermal identity,PDF1,Solyc07g055950,Solyc07g055950.3,Protodermal factor,True,28.198473,13.951530,5.468009e-171,1.220229,-0.031798,86.796117,37.919344
2,Epidermal identity,CD2,Solyc01g091630,Solyc01g091630.3,HD-Zip IV / cuticle regulator,True,17.685753,2.963053,4.498585e-67,0.813039,-0.021187,73.980583,33.987755
3,Epidermal identity,SVB,Solyc01g100750,Solyc01g100750.2,Epidermis/trichome developmental regulator,True,12.919274,4.104692,1.410579e-35,1.168669,-0.030454,39.805825,7.741740
4,Epidermal identity,Wo,Solyc02g080260,Solyc02g080260.3,HD-Zip IV transcription factor,True,6.310112,2.277112,2.424569e-08,0.590130,-0.015378,22.912621,6.790467
5,Trichome morphogenesis,WOX3b,Solyc11g072790,Solyc11g072790.2,WOX transcription factor,True,6.307626,6.004742,2.457852e-08,1.727346,-0.045013,16.699029,0.465516
6,Trichome morphogenesis,MX1 / MIXTA,Solyc01g010910,Solyc01g010910.2,MYB/MIXTA-like transcription factor,True,5.636161,6.286357,1.195652e-06,1.624088,-0.042322,14.951456,0.450336
7,Trichome morphogenesis,MTR2 / CycB3,Solyc06g073990,Solyc06g073990.2,MTR / cyclin-B3-like regulator,True,10.171443,4.616310,6.854180e-22,1.581254,-0.041206,28.155340,2.084704
8,Trichome morphogenesis,MTR3,Solyc01g007870,Solyc01g007870.3,MTR / cyclin-like regulator,True,23.191149,6.688268,1.335790e-115,1.965490,-0.051218,65.825243,8.510854
9,Trichome morphogenesis,HAIR,Solyc10g078970,Solyc10g078970.1,C2H2 zinc-finger protein,False,2.178008,4.794347,3.562463e-01,0.932115,-0.024290,5.825243,0.212518


## Summary

The atlas comparison supports the developmental and glandular-regulatory modules more strongly than the mature metabolic modules.


In [11]:
summary = (
    module_marker_overlap_simple
    .groupby("module_label", sort=False)
    .agg(
        genes=("display_name", "count"),
        atlas_supported=("atlas_trichome_supported", "sum"),
        median_detection_trichome=("pct_detected_trichome", "median"),
    )
)

summary


,genes,atlas_supported,median_detection_trichome
module_label,,,
Epidermal identity,5,5,39.805825
Trichome morphogenesis,6,4,15.825243
Glandular regulation,5,3,16.310680
Type IV acylsugar metabolism,5,1,0.776699
Type VI terpene metabolism,4,1,4.660194
Reference control,7,0,26.213592
